In [26]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import mlflow
import warnings
warnings.filterwarnings('ignore')

In [27]:
import sys
import subprocess
print('kernel python:', sys.executable)
print('python version:', sys.version)
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--user',
        'scikit-learn',
        'mlflow',
        'xgboost',
    ],
    check=True,
)

kernel python: c:\Python312\python.exe
python version: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]


CompletedProcess(args=['c:\\Python312\\python.exe', '-m', 'pip', 'install', '--user', 'scikit-learn', 'mlflow', 'xgboost'], returncode=0)

In [28]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [29]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [30]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": 42,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)


              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [31]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.9560439560439561,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.9159951159951161,
  'support': 300.0}}

In [32]:
import mlflow

In [20]:
# Diagnostic + fix for URL-encoded MLflow paths (handles file:// and sqlite:// URIs)
import os, mlflow, urllib.parse, pathlib

print("ENV MLFLOW_TRACKING_URI =", os.environ.get("MLFLOW_TRACKING_URI"))
current = mlflow.get_tracking_uri()
print("mlflow.get_tracking_uri()   =", current)


def decode_and_fix(uri):
    if not uri:
        return uri
    # file:// local path
    if uri.startswith("file://"):
        raw = uri[len("file://"):]
        if raw.startswith("/"):
            raw = raw.lstrip("/")
        decoded = urllib.parse.unquote(raw)
        p = pathlib.Path(decoded)
        try:
            p.mkdir(parents=True, exist_ok=True)
            print("Ensured directory exists:", p)
        except Exception as e:
            print("Failed to create directory:", e)
            return uri
        new_uri = "file:///" + str(p).replace("\\", "/")
        return new_uri
    # sqlite:///C:/path/to/db
    if uri.startswith("sqlite://"):
        # support both sqlite:/// and sqlite:// formats
        if uri.startswith("sqlite:///"):
            raw = uri[len("sqlite:///"):]
        else:
            raw = uri[len("sqlite://"):]
        decoded = urllib.parse.unquote(raw)
        p = pathlib.Path(decoded)
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            print("Ensured sqlite parent directory exists:", p.parent)
        except Exception as e:
            print("Failed to create sqlite parent directory:", e)
            return uri
        new_uri = "sqlite:///" + str(p).replace("\\", "/")
        return new_uri
    return uri

fixed = decode_and_fix(current)
if fixed != current:
    mlflow.set_tracking_uri(fixed)
    print("Updated mlflow tracking URI to:", mlflow.get_tracking_uri())
else:
    print("No change to mlflow tracking URI.")


ENV MLFLOW_TRACKING_URI = None
mlflow.get_tracking_uri()   = sqlite:///C:/Users/vinay%20agrawal/fullStackDatascience/MachineLearning/MLOPS/mlflow.db
Ensured sqlite parent directory exists: C:\Users\vinay agrawal\fullStackDatascience\MachineLearning\MLOPS
Updated mlflow tracking URI to: sqlite:///C:/Users/vinay agrawal/fullStackDatascience/MachineLearning/MLOPS/mlflow.db


In [21]:
import sqlite3, os

db = r"C:\Users\vinay agrawal\fullStackDatascience\MachineLearning\MLOPS\mlflow.db"
print('db exists:', os.path.exists(db))
if os.path.exists(db):
    conn = sqlite3.connect(db)
    cur = conn.cursor()
    try:
        cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
        print('tables:', [r[0] for r in cur.fetchall()])
        try:
            cur.execute("SELECT version_num FROM alembic_version")
            print('alembic_version:', cur.fetchone())
        except Exception as e:
            print('alembic_version query failed:', e)
    finally:
        conn.close()


db exists: True
tables: ['alembic_version', 'experiment_tags', 'runs', 'latest_metrics', 'metrics', 'inputs', 'input_tags', 'params', 'trace_tags', 'trace_request_metadata', 'tags', 'datasets', 'logged_models', 'logged_model_metrics', 'logged_model_params', 'logged_model_tags', 'assessments', 'spans', 'entity_associations', 'webhook_events', 'scorers', 'scorer_versions', 'evaluation_dataset_tags', 'evaluation_dataset_records', 'endpoint_model_mappings', 'endpoint_bindings', 'endpoint_tags', 'trace_metrics', 'online_scoring_configs', 'span_metrics', 'experiments', 'registered_models', 'model_versions', 'registered_model_tags', 'model_version_tags', 'registered_model_aliases', 'evaluation_datasets', 'webhooks', 'secrets', 'endpoints', 'model_definitions', 'jobs', 'workspaces', 'budget_policies', 'issues', 'guardrails', 'guardrail_configs', 'label_schemas', 'review_queues', 'review_queue_users', 'review_queue_items', 'review_queue_label_schemas', 'mcp_servers', 'mcp_server_versions', 'mcp

In [33]:
mlflow.set_experiment("1st Experiment")
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")


with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metrics({
        'accuracy': report_dict['accuracy'],
        'recall_class_0': report_dict['0']['recall'],
        'recall_class_1': report_dict['1']['recall'],
        'f1_score_macro': report_dict['macro avg']['f1-score'],
    })
    mlflow.sklearn.log_model(lr, "logistic_regression")

2026/08/11 23:20:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run thundering-croc-415 at: http://127.0.0.1:5000/#/experiments/1/runs/c3bece2008304c4a81d5ec18656db387
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
